# 01 — Khám phá dữ liệu (VietNews, WikiLingua-vi)

Notebook này đọc output của phase 1 (`data/processed/<dataset>/<split>.parquet`) để trả lời mấy câu hỏi
trước khi train/đánh giá:

1. Mỗi dataset có bao nhiêu mẫu, văn bản và bản tóm tắt dài bao nhiêu token?
2. Bao nhiêu văn bản vượt ngữ cảnh 1,024 token của ViT5 (lý do cần xử lý văn bản dài)?
3. Bản tóm tắt reference nén mạnh tới đâu, và **trừu tượng** tới đâu (bao nhiêu n-gram không có trong văn bản)?
4. Thông tin của bản tóm tắt nằm ở đâu trong văn bản (lead bias)?
5. Bước lọc câu (`extractive_filter`) cắt bao nhiêu văn bản?

**Cần:** đã chạy `make prepare` (hoặc `make smoke` để có bản nhỏ 200 mẫu), và
`pip install -r requirements-notebooks.txt` trong `.venv`. Notebook chỉ đọc dữ liệu, không ghi gì.

Logic dùng lại từ `vnsum` (cùng tokenizer âm tiết với ROUGE); khi một đoạn phân tích ở đây đáng dùng lại,
chuyển nó thành hàm trong `src/vnsum/` kèm test thay vì sao chép sang notebook khác.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Chạy từ notebooks/ hay từ gốc repo đều được: đường dẫn trong config tính từ gốc repo.
ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
os.chdir(ROOT)

from vnsum.config import load_config  # noqa: E402
from vnsum.eval.metrics import ngrams, novel_ngram_rate, syllables  # noqa: E402

cfg = load_config("configs/data.yaml")
PROCESSED = Path(cfg["paths"]["processed_dir"])
LONG = int(cfg["stats"]["long_doc_threshold_tokens"])  # 1024
DATASETS = [name for name, d in cfg["datasets"].items() if d.get("enabled", True)]
SPLITS = ["train", "validation", "test"]
SAMPLE = 3000  # số mẫu / dataset cho các phép tính chậm (n-gram); tăng nếu muốn chính xác hơn
SEED = int(cfg["seed"])

pd.set_option("display.max_colwidth", 200)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3})
print("Gốc repo:", ROOT)
print("Dataset:", DATASETS, "| ngưỡng văn bản dài:", LONG, "token")

## 1. Đọc dữ liệu

In [ ]:
COLUMNS = ["id", "document", "summary", "document_tokens", "summary_tokens",
           "input_tokens", "was_filtered", "was_truncated"]

frames = []
for name in DATASETS:
    for split in SPLITS:
        path = PROCESSED / name / f"{split}.parquet"
        if not path.exists():
            print(f"(bỏ qua) chưa có {path}")
            continue
        df = pd.read_parquet(path)
        frames.append(df[[c for c in COLUMNS if c in df.columns]].assign(dataset=name, split=split))
if not frames:
    raise FileNotFoundError(f"Không có file parquet nào trong {PROCESSED}. Chạy `make smoke` hoặc `make prepare` trước.")
data = pd.concat(frames, ignore_index=True)
data["is_long"] = data["document_tokens"] > LONG

data.pivot_table(index="dataset", columns="split", values="id", aggfunc="count", margins=True, margins_name="tổng")

## 2. Độ dài văn bản và bản tóm tắt (token ViT5)

Trục x dạng log vì độ dài lệch phải rất mạnh. Đường đứt là ngưỡng 1,024 token: phần bên phải là văn bản
ViT5 không đọc hết được trong một lần.

In [ ]:
fig, axes = plt.subplots(1, 2)
for name, g in data.groupby("dataset"):
    bins = np.logspace(np.log10(max(g["document_tokens"].min(), 1)), np.log10(g["document_tokens"].max() + 1), 60)
    axes[0].hist(g["document_tokens"], bins=bins, alpha=0.5, label=name)
    axes[1].hist(g["summary_tokens"], bins=40, alpha=0.5, label=name)
axes[0].axvline(LONG, color="k", ls="--", lw=1)
axes[0].set(xscale="log", title="Văn bản", xlabel="token")
axes[1].set(title="Bản tóm tắt", xlabel="token")
axes[0].legend()
plt.tight_layout()
plt.show()

(data.groupby("dataset")
     .agg(n=("id", "count"),
          doc_median=("document_tokens", "median"),
          doc_p95=("document_tokens", lambda s: s.quantile(0.95)),
          sum_median=("summary_tokens", "median"),
          sum_p95=("summary_tokens", lambda s: s.quantile(0.95)),
          pct_long=("is_long", lambda s: 100 * s.mean()))
     .round(1))

## 3. Tỷ lệ nén

`document_tokens / summary_tokens`: bản tóm tắt ngắn hơn văn bản bao nhiêu lần. Tỷ lệ càng cao, model càng
phải chọn lọc nhiều.

In [ ]:
data["compression"] = data["document_tokens"] / data["summary_tokens"].clip(lower=1)
ax = data.boxplot(column="compression", by="dataset", showfliers=False)
ax.set(title="", ylabel="lần", xlabel="")
plt.suptitle("Tỷ lệ nén (không vẽ outlier)")
plt.show()
data.groupby("dataset")["compression"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(1)

## 4. Độ trừu tượng của reference

Tỷ lệ n-gram của bản tóm tắt **không** xuất hiện trong văn bản (cùng định nghĩa với cột `novel_*gram` trong
`results/main/per_sample.csv`, nên so được trực tiếp với output của model). Reference càng trừu tượng thì
model chỉ biết chép câu càng khó đạt ROUGE cao.

In [ ]:
sample = (data[data["split"] == "train"] if (data["split"] == "train").any() else data)
sample = pd.concat([g.sample(min(len(g), SAMPLE), random_state=SEED) for _, g in sample.groupby("dataset")])
for n in (1, 2, 3):
    sample[f"novel_{n}gram"] = [novel_ngram_rate(s, d, n) for s, d in zip(sample["summary"], sample["document"])]

novel_cols = ["novel_1gram", "novel_2gram", "novel_3gram"]
sample.groupby("dataset")[novel_cols].mean().round(1).plot.bar(rot=0, title="% n-gram mới trong reference (trung bình)")
plt.ylabel("%")
plt.show()
sample.groupby("dataset")[novel_cols].mean().round(1)

## 5. Thông tin của bản tóm tắt nằm ở đâu trong văn bản?

Với mỗi bigram của bản tóm tắt có trong văn bản, lấy vị trí xuất hiện đầu tiên (0 = đầu bài, 1 = cuối bài).
Nếu phân bố dồn về bên trái thì dataset có **lead bias**: chép vài câu đầu đã được điểm khá (Lead-3), và
cắt bỏ phần cuối văn bản dài ít mất thông tin.

In [ ]:
def bigram_positions(summary: str, document: str) -> list[float]:
    doc = syllables(document)
    if len(doc) < 2:
        return []
    first: dict[tuple[str, ...], int] = {}
    for i, g in enumerate(ngrams(doc, 2)):
        first.setdefault(g, i)
    span = max(len(doc) - 2, 1)
    return [first[g] / span for g in set(ngrams(syllables(summary), 2)) if g in first]

fig, ax = plt.subplots()
for name, g in sample.groupby("dataset"):
    pos = [p for s, d in zip(g["summary"], g["document"]) for p in bigram_positions(s, d)]
    ax.hist(pos, bins=20, range=(0, 1), density=True, alpha=0.5, label=f"{name} (trong 20% đầu: {100 * np.mean(np.array(pos) < 0.2):.0f}%)")
ax.axhline(1, color="k", ls="--", lw=1)  # phân bố đều
ax.set(title="Vị trí bigram của reference trong văn bản", xlabel="vị trí tương đối", ylabel="mật độ")
ax.legend()
plt.show()

## 6. Bước lọc câu cho văn bản dài (`extractive_filter`)

Văn bản vượt ngân sách token được lọc câu (BM25) xuống ≤ 1,024 token; `was_truncated` là trường hợp lọc
xong vẫn phải cắt. Chỉ có khi phase 1 chạy với `long_context.strategy: extractive_filter`.

In [ ]:
if {"was_filtered", "was_truncated", "input_tokens"} <= set(data.columns):
    summary_lc = (data.groupby(["dataset", "split"])
                      .agg(n=("id", "count"),
                           pct_filtered=("was_filtered", lambda s: 100 * s.mean()),
                           pct_truncated=("was_truncated", lambda s: 100 * s.mean()),
                           input_tokens_max=("input_tokens", "max"))
                      .round(2))
    display(summary_lc)  # noqa: F821 (display có sẵn trong Jupyter)
    kept = data.loc[data["was_filtered"], "input_tokens"] / data.loc[data["was_filtered"], "document_tokens"]
    print(f"Văn bản bị lọc giữ lại trung bình {100 * kept.mean():.0f}% số token gốc.")
else:
    print("Dữ liệu không có cột của extractive_filter (phase 1 chạy với strategy khác).")

## 7. Xem vài ví dụ

Đọc tận mắt vẫn là cách tốt nhất để hiểu dữ liệu. Đổi `random_state` để xem ví dụ khác.

In [ ]:
cols = ["dataset", "document_tokens", "summary_tokens", "summary", "document"]
examples = pd.concat([g.sample(min(len(g), 2), random_state=1) for _, g in data.groupby("dataset")])[cols]
examples.assign(document=examples["document"].str.slice(0, 400) + " …")

## Ghi chú

- Số liệu tổng hợp chính thức của phase 1 nằm ở `data/processed/stats.json` (do `scripts/prepare_data.py` ghi);
  notebook này để khám phá, không thay thế file đó.
- Ý tưởng tiếp theo: so độ trừu tượng ở mục 4 với cột `novel_*gram` của model trong
  `results/main/per_sample.csv` (xem `02_error_analysis.ipynb`).